# PHM North America 2025 — Manutenzione predittiva motori jet

Predizione dei **cicli residui** a tre eventi di manutenzione:

- **WW** — lavaggio del compressore (`Cycles_to_WW`)
- **HPC_SV** — guasto del compressore (`Cycles_to_HPC_SV`)
- **HPT_SV** — guasto della turbina (`Cycles_to_HPT_SV`)

**Punteggio ufficiale leaderboard: 71.** Questo notebook è il refactor pulito
della pipeline finale (catena *Domino*); il comportamento numerico è identico
all'originale `prova3Nic.ipynb` (cella con `main()`).

## Ordine di esecuzione
Il notebook è pensato per **Restart & Run All**: le celle vanno eseguite
dall'alto verso il basso. L'ordine delle sezioni è:

1. Setup e import
2. Configurazione (percorsi, iperparametri, colonne, costanti)
3. Preprocessing e feature fisiche
4. Preparazione dati per evento
5. Metrica e ottimizzazione margini
6. Training — catena Domino
7. Caricamento dati e addestramento
8. Validazione (submission su `data/val/`)
9. Generazione submission finale (submission su `data/test/`)
10. (Opzionale) Importanza delle feature

## Dati richiesti
- **Training**: `data_elaborated/train/train_cleaned.csv`
  (fallback: `data_elaborated/train/train_with_physics_residuals.csv`)
- **Validazione**: cartella `data/val/` (un CSV per motore)
- **Test/gara**: cartella `data/test/` (un CSV per motore)
- **Output**: cartella `risultati/` (creata se assente)

## Note su possibili leakage (comportamento invariato, solo segnalato)
- Nel training della catena, se manca la predizione OOF di un evento a monte
  (disallineamento dei filtri di regime), viene sostituita col **valore reale
  del target** (`fillna(...)`): potenziale *target leakage* — vedi `# TODO` nella
  Sezione 6.
- I margini `a·pred + b` sono ottimizzati sugli **stessi** dati out-of-fold su cui
  vengono misurati: la stima interna può essere leggermente ottimistica.

## 1. Setup e import

In [ ]:
import os
import re
import glob
from typing import Optional

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from scipy.optimize import minimize

## 2. Configurazione

Tutti i parametri della pipeline in un unico posto: percorsi, seed, iperparametri
dei modelli, colonne usate, costanti fisiche e soglie dei filtri di regime.
Modificare qui — nessun valore "magico" sparso nel resto del notebook.

In [ ]:
# --- Riproducibilità -------------------------------------------------------
RANDOM_STATE: int = 42

# --- Percorsi --------------------------------------------------------------
DATA_ELAB_TRAIN_DIR = os.path.join("data_elaborated", "train")
TRAIN_PRIMARY   = os.path.join(DATA_ELAB_TRAIN_DIR, "train_cleaned.csv")
TRAIN_FALLBACK  = os.path.join(DATA_ELAB_TRAIN_DIR, "train_with_physics_residuals.csv")
VAL_DIR         = os.path.join("data", "val") + os.sep      # motori di validazione
TEST_DIR        = os.path.join("data", "test") + os.sep     # motori di test/gara
RESULTS_DIR     = "risultati"
SUB_VAL_FILE    = os.path.join(RESULTS_DIR, "submission_val_Domino.csv")
SUB_FINAL_FILE  = os.path.join(RESULTS_DIR, "submission_final.csv")

# --- Target ----------------------------------------------------------------
# NB: l'ordine è preservato dall'originale (influenza l'ordine delle colonne).
TARGETS = ["Cycles_to_HPC_SV", "Cycles_to_HPT_SV", "Cycles_to_WW"]

# --- Costanti fisiche (ciclo di Brayton, unità imperiali) ------------------
STD_TEMP_R = 518.67    # temperatura standard [Rankine]
STD_PRES   = 14.696    # pressione standard [psia]
GAMMA_AIR  = 1.4       # rapporto dei calori specifici dell'aria

# --- Soglie dei filtri di regime stabile -----------------------------------
ALT_THRESHOLD_MECH   = 20000   # crociera: Sensed_Altitude > 20000 (HPC/HPT)
SPEED_THRESHOLD_WASH = 8000    # alta potenza: Sensed_Core_Speed > 8000 (WW)

# --- Feature per evento ----------------------------------------------------
# Modelli meccanici (HPC / HPT)
MECH_PHY_COLS = ["Phy_T45_Corr", "Phy_Compressor_Eff", "Phy_Heat_Index", "Phy_Core_Speed_Corr"]
MECH_RAW_COLS = ["Sensed_Ps3", "Sensed_T3"]
MECH_ROLL_WINDOW   = 10   # finestra media mobile (smooth/std)
MECH_TREND_PERIODS = 5    # passi per la differenza (trend)

# Modello lavaggio (WW)
WASH_PHY_COLS = ["Phy_Compressor_Eff", "Phy_Heat_Index", "Phy_WFuel_Corr"]
WASH_RAW_COLS = ["Sensed_WFuel", "Sensed_T45"]
WASH_ROLL_WINDOW = 5   # finestra media mobile (smooth/std)
WASH_N_LAGS      = 3   # numero di lag temporali (lag1..lag3)

# --- Metrica ufficiale -----------------------------------------------------
SCORE_ALPHA     = 0.01   # decadimento del peso col residuo vero
SCORE_BETA_WW   = 1.0    # numeratore beta per WW  (beta = 1/max_train_val)
SCORE_BETA_MECH = 2.0    # numeratore beta per HPC/HPT (beta = 2/max_train_val)
SCORE_W_OVER    = 2.0    # peso quando si sovrastima (error >= 0), penalizza di più
SCORE_W_UNDER   = 1.0    # peso quando si sottostima (error < 0)

# --- Validazione incrociata e ottimizzazione margini -----------------------
GKF_SPLITS   = 4                # Group K-Fold per motore (ESN)
MARGIN_INIT  = [0.95, -50.0]    # punto iniziale Nelder-Mead per (a, b)

# --- Iperparametri dei modelli (GradientBoosting, Huber loss) --------------
# HPC e HPT condividono gli stessi iperparametri (come nell'originale).
MODEL_PARAMS = {
    "WW":  {"loss": "huber", "n_estimators": 300, "max_depth": 6,
            "learning_rate": 0.05, "subsample": 1.0, "random_state": RANDOM_STATE},
    "HPC": {"loss": "huber", "n_estimators": 500, "max_depth": 4,
            "learning_rate": 0.03, "subsample": 0.7, "random_state": RANDOM_STATE},
    "HPT": {"loss": "huber", "n_estimators": 500, "max_depth": 4,
            "learning_rate": 0.03, "subsample": 0.7, "random_state": RANDOM_STATE},
}

# --- Mappa di armonizzazione dei nomi sensori (train vs gara) ---------------
SENSOR_RENAME_MAP = {
    "Cycles": "Cycles_Since_New", "Cycle": "Cycles_Since_New",
    "Altitude": "Sensed_Altitude", "Mach": "Sensed_Mach",
    "TRA": "Sensed_TRA", "T2": "Sensed_T2", "T24": "Sensed_T24",
    "T25": "Sensed_T25", "Pt2": "Sensed_Pt2",
    "W": "Sensed_WFuel", "WFuel": "Sensed_WFuel",
    "Core_Speed": "Sensed_Core_Speed", "N2": "Sensed_Core_Speed",
    "Fan_Speed": "Sensed_Fan_Speed", "N1": "Sensed_Fan_Speed",
    "T30": "Sensed_T3", "T3": "Sensed_T3",
    "T48": "Sensed_T45", "T45": "Sensed_T45", "T50": "Sensed_T5",
    "P15": "Sensed_P15", "P2": "Sensed_P2", "P21": "Sensed_P21",
    "P24": "Sensed_P24", "P25": "Sensed_P25",
    "Ps30": "Sensed_Ps3", "Ps3": "Sensed_Ps3", "P40": "Sensed_P40",
    "P50": "Sensed_P50", "HPC_SV": "Cycles_to_HPC_SV",
    "HPT_SV": "Cycles_to_HPT_SV", "WW": "Cycles_to_WW",
}
# Colonne che NON vanno prefissate con 'Sensed_' durante l'armonizzazione.
ID_COLS = ["ESN", "Cycles_Since_New", "Snapshot", "File_ID", "file"]

# --- Controlli robusti sui percorsi ----------------------------------------
os.makedirs(RESULTS_DIR, exist_ok=True)
if not (os.path.exists(TRAIN_PRIMARY) or os.path.exists(TRAIN_FALLBACK)):
    print(f"[ATTENZIONE] Nessun file di training trovato in '{DATA_ELAB_TRAIN_DIR}'.")
for _d in (VAL_DIR, TEST_DIR):
    if not os.path.isdir(_d):
        print(f"[ATTENZIONE] Cartella dati assente: '{_d}' (la relativa submission verrà saltata).")

## 3. Preprocessing e feature fisiche

`harmonize_columns` uniforma i nomi dei sensori tra file di training e di gara.
`add_physics_features` calcola le feature basate sul ciclo di Brayton, corrette
per le condizioni di volo (θ, δ): efficienza isentropica del compressore,
heat index, e versioni corrette di velocità/carburante/temperatura.

In [ ]:
def harmonize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Uniforma i nomi delle colonne tra file di training e di gara.

    Applica ``SENSOR_RENAME_MAP`` e poi prefissa con ``Sensed_`` ogni colonna
    non-identificativa che non abbia già un prefisso noto
    (``Sensed_``/``Phy_``/``Cycles_to_``). La funzione è idempotente.
    """
    df = df.rename(columns=SENSOR_RENAME_MAP)
    new_cols = {}
    for col in df.columns:
        if col not in ID_COLS:
            if not col.startswith("Sensed_") and not col.startswith("Phy_") and not col.startswith("Cycles_to_"):
                new_cols[col] = f"Sensed_{col}"
    if new_cols:
        df = df.rename(columns=new_cols)
    return df


def add_physics_features(df: pd.DataFrame) -> pd.DataFrame:
    """Aggiunge le feature fisiche (ciclo di Brayton) corrette per il volo.

    Calcola i parametri corretti θ (temperatura) e δ (pressione) e da questi:
    velocità/carburante/temperatura corretti, efficienza isentropica del
    compressore (``Phy_Compressor_Eff``) e heat index (``Phy_Heat_Index``).
    """
    df = df.copy()

    if "Sensed_T25" in df.columns and "Sensed_Pt2" in df.columns:
        theta = df["Sensed_T25"] / STD_TEMP_R
        theta = np.maximum(theta, 0.0001)
        delta = df["Sensed_Pt2"] / STD_PRES
        theta = theta.replace(0, 1); delta = delta.replace(0, 1)

        if "Sensed_Core_Speed" in df.columns:
            df["Phy_Core_Speed_Corr"] = df["Sensed_Core_Speed"] / np.sqrt(theta)
        if "Sensed_WFuel" in df.columns:
            df["Phy_WFuel_Corr"] = df["Sensed_WFuel"] / (delta * np.sqrt(theta))
        if "Sensed_T45" in df.columns:
            df["Phy_T45_Corr"] = df["Sensed_T45"] / theta

        if "Sensed_T3" in df.columns and "Sensed_Ps3" in df.columns:
            T_in_R = df["Sensed_T25"]
            T_out_R = df["Sensed_T3"]
            P_in = df["Sensed_P25"] if "Sensed_P25" in df.columns else df["Sensed_Pt2"]
            P_out = df["Sensed_Ps3"]
            pr = P_out / P_in
            k = (GAMMA_AIR - 1) / GAMMA_AIR
            T_iso_R = T_in_R * (pr ** k)
            df["Phy_Compressor_Eff"] = (T_iso_R - T_in_R) / (T_out_R - T_in_R)

    if "Sensed_T45" in df.columns and "Sensed_Ps3" in df.columns:
        df["Phy_Heat_Index"] = df["Sensed_T45"] / df["Sensed_Ps3"]
    return df

## 4. Preparazione dati per evento

`prepare_mechanical_data` (HPC/HPT) filtra il regime di crociera
(`Sensed_Altitude > 20000`); `prepare_wash_data` (WW) filtra l'alta potenza
(`Sensed_Core_Speed > 8000`). Entrambe aggregano per ciclo di volo e aggiungono
feature temporali (medie mobili, trend, lag, cicli dall'ultimo lavaggio).
Il parametro `is_test` distingue il caso multi-motore (training, group-by `ESN`)
dal caso singolo motore (inferenza).

In [ ]:
def prepare_mechanical_data(df: pd.DataFrame, is_test: bool = False) -> pd.DataFrame:
    """Prepara i dati per i modelli meccanici (HPC / HPT).

    Filtra il regime di crociera, aggrega per ciclo di volo (media dei sensori
    fisici) e aggiunge feature temporali (media mobile ``_smooth``, trend
    ``_trend`` come differenza a ``MECH_TREND_PERIODS`` passi, e ``_std``).
    """
    df = df.copy()
    df = harmonize_columns(df)
    if "Sensed_Altitude" in df.columns:
        df = df[df["Sensed_Altitude"] > ALT_THRESHOLD_MECH].copy()
    df = add_physics_features(df)

    use_cols = [c for c in MECH_PHY_COLS + MECH_RAW_COLS if c in df.columns]

    agg_dict = {col: "mean" for col in use_cols}
    # Aggreghiamo tutti i target (servono per la catena Domino / multi-task).
    for t in TARGETS:
        if t in df.columns:
            agg_dict[t] = "first"

    if not is_test and "ESN" in df.columns:
        df_grouped = df.groupby(["ESN", "Cycles_Since_New"]).agg(agg_dict).reset_index()
        df_grouped = df_grouped.sort_values(["ESN", "Cycles_Since_New"])
        for col in use_cols:
            df_grouped[f"{col}_smooth"] = df_grouped.groupby("ESN")[col].transform(lambda x: x.rolling(window=MECH_ROLL_WINDOW, min_periods=1).mean())
            df_grouped[f"{col}_trend"] = df_grouped.groupby("ESN")[f"{col}_smooth"].diff(periods=MECH_TREND_PERIODS).fillna(0)
            df_grouped[f"{col}_std"] = df_grouped.groupby("ESN")[col].transform(lambda x: x.rolling(window=MECH_ROLL_WINDOW, min_periods=1).std()).fillna(0)
    else:
        df_grouped = df.groupby("Cycles_Since_New").agg(agg_dict).reset_index()
        df_grouped = df_grouped.sort_values("Cycles_Since_New")
        for col in use_cols:
            df_grouped[f"{col}_smooth"] = df_grouped[col].rolling(window=MECH_ROLL_WINDOW, min_periods=1).mean()
            df_grouped[f"{col}_trend"] = df_grouped[f"{col}_smooth"].diff(periods=MECH_TREND_PERIODS).fillna(0)
            df_grouped[f"{col}_std"] = df_grouped[col].rolling(window=MECH_ROLL_WINDOW, min_periods=1).std().fillna(0)

    return df_grouped.ffill().bfill().fillna(0)


def prepare_wash_data(df: pd.DataFrame, is_test: bool = False) -> pd.DataFrame:
    """Prepara i dati per il modello di lavaggio (WW).

    Filtra il regime di alta potenza, aggrega per ciclo di volo (``mean``/``max``)
    e aggiunge feature temporali: lag (``_lag1..lag{WASH_N_LAGS}``), media mobile
    ``_smooth``, differenza ``_diff``, deviazione ``_std`` e, se disponibile,
    ``Cycles_Since_Last_Wash`` (cicli dall'ultimo lavaggio).
    """
    df = df.copy()
    df = harmonize_columns(df)
    if "Sensed_Core_Speed" in df.columns:
        df = df[df["Sensed_Core_Speed"] > SPEED_THRESHOLD_WASH].copy()
    df = add_physics_features(df)

    use_cols = [c for c in WASH_PHY_COLS + WASH_RAW_COLS if c in df.columns]

    agg_dict = {}
    for c in use_cols:
        agg_dict[c] = ["mean", "max"]
    if "Cycles_to_WW" in df.columns:
        agg_dict["Cycles_to_WW"] = "first"
    if "Cumulative_WWs" in df.columns:
        agg_dict["Cumulative_WWs"] = "max"

    if not is_test and "ESN" in df.columns:
        df_grouped = df.groupby(["ESN", "Cycles_Since_New"]).agg(agg_dict)
    else:
        df_grouped = df.groupby("Cycles_Since_New").agg(agg_dict)

    new_cols = []
    feature_cols = []
    for c, s in df_grouped.columns:
        if c in ["Cycles_to_WW", "Cumulative_WWs"] or s == "":
            new_cols.append(c)
        else:
            name = f"{c}_{s}"
            new_cols.append(name)
            feature_cols.append(name)

    df_grouped.columns = new_cols
    df_grouped = df_grouped.reset_index()

    if not is_test and "ESN" in df.columns:
        df_grouped = df_grouped.sort_values(["ESN", "Cycles_Since_New"])
        if "Cumulative_WWs" in df_grouped.columns:
            df_grouped["WW_Change"] = df_grouped.groupby("ESN")["Cumulative_WWs"].diff().fillna(0)
            df_grouped["Wash_Session_ID"] = df_grouped.groupby("ESN")["WW_Change"].cumsum()
            df_grouped["Cycles_Since_Last_Wash"] = df_grouped.groupby(["ESN", "Wash_Session_ID"]).cumcount()
            feature_cols.append("Cycles_Since_Last_Wash")

        for col in feature_cols:
            if col == "Cycles_Since_Last_Wash":
                continue
            for i in range(1, WASH_N_LAGS + 1):
                df_grouped[f"{col}_lag{i}"] = df_grouped.groupby("ESN")[col].shift(i)
            df_grouped[f"{col}_smooth"] = df_grouped.groupby("ESN")[col].transform(lambda x: x.rolling(window=WASH_ROLL_WINDOW, min_periods=1).mean())
            df_grouped[f"{col}_diff"] = df_grouped.groupby("ESN")[col].diff()
            df_grouped[f"{col}_std"] = df_grouped.groupby("ESN")[col].transform(lambda x: x.rolling(window=WASH_ROLL_WINDOW, min_periods=1).std()).fillna(0)
    else:
        df_grouped = df_grouped.sort_values("Cycles_Since_New")
        if "Cumulative_WWs" in df_grouped.columns:
            df_grouped["WW_Change"] = df_grouped["Cumulative_WWs"].diff().fillna(0)
            df_grouped["Wash_Session_ID"] = df_grouped["WW_Change"].cumsum()
            df_grouped["Cycles_Since_Last_Wash"] = df_grouped.groupby("Wash_Session_ID").cumcount()
            feature_cols.append("Cycles_Since_Last_Wash")

        for col in feature_cols:
            if col == "Cycles_Since_Last_Wash":
                continue
            for i in range(1, WASH_N_LAGS + 1):
                df_grouped[f"{col}_lag{i}"] = df_grouped[col].shift(i)
            df_grouped[f"{col}_smooth"] = df_grouped[col].rolling(window=WASH_ROLL_WINDOW, min_periods=1).mean()
            df_grouped[f"{col}_diff"] = df_grouped[col].diff()
            df_grouped[f"{col}_std"] = df_grouped[col].rolling(window=WASH_ROLL_WINDOW, min_periods=1).std().fillna(0)

    df_grouped = df_grouped.ffill().bfill().fillna(0)
    drop_cols = ["WW_Change", "Wash_Session_ID", "Cumulative_WWs"]
    return df_grouped.drop(columns=[c for c in drop_cols if c in df_grouped.columns])

## 5. Metrica ufficiale e ottimizzazione dei margini

`get_exact_competition_score` implementa la funzione di punteggio della gara
(asimmetrica: penalizza di più la sovrastima). `optimize_margins_oof` addestra
il modello in Group K-Fold per motore, produce predizioni **out-of-fold** (OOF)
e ottimizza con Nelder-Mead un margine di sicurezza `a·pred + b` che minimizza
il punteggio ufficiale valutato OOF.

In [ ]:
def get_exact_competition_score(y_true: np.ndarray, y_pred: np.ndarray,
                                component_type: str, max_train_val: float) -> float:
    """Punteggio ufficiale della competizione (più basso = meglio).

    Errore asimmetrico: la sovrastima (``error >= 0``) pesa il doppio della
    sottostima. ``beta`` dipende dal tipo di componente e normalizza rispetto al
    massimo valore del target nel training.
    """
    error = y_pred - y_true
    alpha = SCORE_ALPHA
    beta = (SCORE_BETA_WW / max_train_val) if component_type == "WW" else (SCORE_BETA_MECH / max_train_val)
    w = np.where(error >= 0, SCORE_W_OVER / (1.0 + alpha * y_true), SCORE_W_UNDER / (1.0 + alpha * y_true))
    return np.mean(w * (error ** 2) * beta)


def optimize_margins_oof(X_df: pd.DataFrame, y_series: pd.Series, groups: pd.Series,
                         model_params: dict, ctype: str, max_train_val: float):
    """Addestra in Group K-Fold, calcola le predizioni OOF e ottimizza i margini.

    Restituisce ``(final_model, best_a, best_b, oof_preds)`` dove ``final_model`` è
    riaddestrato su tutti i dati, ``(best_a, best_b)`` è il margine ``a·pred + b``
    ottimizzato via Nelder-Mead, e ``oof_preds`` sono le predizioni out-of-fold
    **pure** (senza margine) — usate come feature dai modelli a valle (Domino).
    """
    gkf = GroupKFold(n_splits=GKF_SPLITS)
    oof_preds = np.zeros(len(X_df))

    X_arr = X_df.values
    y_arr = y_series.values
    groups_arr = groups.values

    for train_idx, val_idx in gkf.split(X_arr, y_arr, groups=groups_arr):
        X_tr, y_tr = X_arr[train_idx], y_arr[train_idx]
        X_v = X_arr[val_idx]

        model = GradientBoostingRegressor(**model_params)
        model.fit(X_tr, y_tr)
        oof_preds[val_idx] = model.predict(X_v)

    def objective(params):
        a, b = params
        y_adj = np.maximum(0, oof_preds * a + b)
        return get_exact_competition_score(y_arr, y_adj, ctype, max_train_val)

    # TODO [leakage noto]: i margini (a, b) sono ottimizzati sugli stessi OOF su cui
    #      poi vengono misurati -> la stima OOF del punteggio può essere ottimistica.
    res = minimize(objective, x0=MARGIN_INIT, method="Nelder-Mead")
    best_a, best_b = res.x

    score_base = get_exact_competition_score(y_arr, np.maximum(0, oof_preds), ctype, max_train_val)
    print(f"    [OOF] Score Pieno (Senza Margini): {score_base:.4f}")
    print(f"    [OOF] Score Ottimizzato (Con Margini): {res.fun:.4f}")
    print(f"    [OOF] Moltiplicatore Trovato: {best_a:.4f} | Offset: {best_b:.2f}")

    final_model = GradientBoostingRegressor(**model_params)
    final_model.fit(X_arr, y_arr)

    # Restituiamo anche le predizioni PURE per passarle ai modelli successivi.
    return final_model, best_a, best_b, oof_preds

## 6. Training — catena *Domino*

I tre eventi sono termodinamicamente collegati, quindi i modelli vengono
addestrati **in cascata**:

1. **WW** — addestrato per primo; le sue predizioni OOF diventano una feature.
2. **HPC** — usa `Pred_WW` (OOF) come input.
3. **HPT** — usa `Pred_WW` e `Pred_HPC` (OOF) come input.

Lo stesso ordine è replicato in inferenza (Sezioni 8–9).

In [ ]:
def train_domino_chain(df_mech: pd.DataFrame, df_wash: pd.DataFrame):
    """Addestra la catena Domino WW -> HPC -> HPT sulle predizioni OOF.

    Per ogni evento inietta come feature le predizioni OOF degli eventi a monte,
    ottimizza i margini e registra le OOF (mappate per ``(ESN, Cycles_Since_New)``)
    per il modello successivo.

    Restituisce ``(trained_models, feature_sets, oof_predictions_dict)``.
    """
    # ORDINE TASSATIVO: WW passa l'informazione a HPC, e poi a HPT.
    tasks = [
        {"target": "Cycles_to_WW",     "data": df_wash, "type": "WW"},
        {"target": "Cycles_to_HPC_SV", "data": df_mech, "type": "HPC"},
        {"target": "Cycles_to_HPT_SV", "data": df_mech, "type": "HPT"},
    ]

    trained_models = {}
    feature_sets = {}
    oof_predictions_dict = {}   # predizioni OOF "cieche" per la catena

    print("\n============================================================")
    print(" AVVIO TRAINING MULTI-TASK (EFFETTO DOMINO & OOF OPTIMIZATION)")
    print("============================================================")

    for task in tasks:
        target = task["target"]
        df = task["data"].copy()
        ctype = task["type"]

        if target not in df.columns or "ESN" not in df.columns:
            continue

        # INIEZIONE DELLE FEATURE OOF (costruzione della catena del degrado).
        if ctype == "HPC":
            df = df.set_index(["ESN", "Cycles_Since_New"])
            df["Pred_WW"] = oof_predictions_dict["WW"]
            df = df.reset_index()
            # TODO [leakage noto]: se manca l'OOF per asimmetria dei filtri, si usa il
            #      target reale come paracadute -> potenziale target leakage da quantificare.
            df["Pred_WW"] = df["Pred_WW"].fillna(df["Cycles_to_WW"])

        elif ctype == "HPT":
            df = df.set_index(["ESN", "Cycles_Since_New"])
            df["Pred_WW"] = oof_predictions_dict["WW"]
            df["Pred_HPC"] = oof_predictions_dict["HPC"]
            df = df.reset_index()
            # TODO [leakage noto]: come sopra, paracadute col target reale.
            df["Pred_WW"] = df["Pred_WW"].fillna(df["Cycles_to_WW"])
            df["Pred_HPC"] = df["Pred_HPC"].fillna(df["Cycles_to_HPC_SV"])

        # Selezione feature (include automaticamente Pred_WW e Pred_HPC).
        features = [c for c in df.columns if c not in ["ESN", target, "Snapshot"] and "Cycles_to_" not in c]
        feature_sets[ctype] = features

        X = df[features]
        y = df[target]
        groups = df["ESN"]
        max_train_val = y.max()

        imputer = SimpleImputer(strategy="mean")
        X_clean = pd.DataFrame(imputer.fit_transform(X), columns=features)

        print(f"\n>>> Modello {ctype} (Feats: {len(features)})")

        params = MODEL_PARAMS[ctype]

        final_model, best_a, best_b, oof_preds = optimize_margins_oof(
            X_clean, y, groups, params, ctype, max_train_val
        )

        # Registriamo l'OOF per i modelli a valle, mappata per (ESN, ciclo).
        oof_series = pd.Series(
            oof_preds,
            index=pd.MultiIndex.from_arrays([df["ESN"], df["Cycles_Since_New"]]),
        )
        oof_predictions_dict[ctype] = oof_series

        trained_models[ctype] = {
            "model": final_model,
            "imputer": imputer,
            "a": best_a,
            "b": best_b,
        }

    return trained_models, feature_sets, oof_predictions_dict

### Funzione di inferenza (effetto Domino in gara)

`generate_submission_file` applica la catena a ogni motore della cartella
indicata, rispettando l'ordine **WW → HPC → HPT** e iniettando le predizioni
pure a monte come feature. Salva il CSV di submission nel formato richiesto.

In [ ]:
def generate_submission_file(folder_path: str, models_dict: dict, feature_sets: dict,
                             output_file: str = "risultati/submission.csv") -> None:
    """Genera il file di submission applicando la catena Domino in inferenza.

    Per ogni CSV motore nella cartella, prepara i dati meccanici e di lavaggio,
    predice nell'ordine WW -> HPC -> HPT iniettando le predizioni pure a monte,
    applica il margine di sicurezza e scrive il CSV finale (cicli residui interi).
    """
    search_path = os.path.join(folder_path, "*.csv")
    all_files = glob.glob(search_path)

    def extract_number(filepath: str) -> int:
        match = re.search(r"\d+", os.path.basename(filepath))
        return int(match.group()) if match else 0

    all_files_sorted = sorted(all_files, key=extract_number)
    results = []
    print(f"\n--- GENERAZIONE SUBMISSION FILE ({len(all_files_sorted)} files) ---")

    for file_path in all_files_sorted:
        file_id = os.path.basename(file_path).replace(".csv", "")

        try:
            df_test_raw = pd.read_csv(file_path)
        except Exception:
            continue
        if df_test_raw.empty:
            continue

        df_test_raw = harmonize_columns(df_test_raw)
        df_mech = prepare_mechanical_data(df_test_raw, is_test=True)
        df_wash = prepare_wash_data(df_test_raw, is_test=True)

        data_map = {"HPT": df_mech, "HPC": df_mech, "WW": df_wash}
        row_pred = {"file": file_id}
        pure_preds = {}   # predizioni pure per lo stacking

        # ORDINE TASSATIVO PER EFFETTO DOMINO: prima WW, poi HPC, infine HPT.
        for ctype in ["WW", "HPC", "HPT"]:
            model = models_dict[ctype]["model"]
            best_a = models_dict[ctype]["a"]
            best_b = models_dict[ctype]["b"]
            imputer = models_dict[ctype]["imputer"]
            df = data_map[ctype].copy()
            features = feature_sets[ctype]

            if df.empty:
                row_pred[ctype] = 0; pure_preds[ctype] = 0; continue

            # INIEZIONE DELLE FEATURE PREDETTE (stacking).
            if ctype == "HPC":
                df["Pred_WW"] = pure_preds.get("WW", 0)
            elif ctype == "HPT":
                df["Pred_WW"] = pure_preds.get("WW", 0)
                df["Pred_HPC"] = pure_preds.get("HPC", 0)

            for f in features:
                if f not in df.columns:
                    df[f] = 0

            X_test = df[features].replace([np.inf, -np.inf], np.nan)

            try:
                X_clean = pd.DataFrame(imputer.transform(X_test), columns=features).fillna(0)

                # LA PREDIZIONE (si usa l'ultimo ciclo disponibile).
                pred_raw = model.predict(X_clean.values)[-1]
                pred_safe = (pred_raw * best_a) + best_b

                pure_preds[ctype] = pred_raw          # "verità" per il modello successivo
                row_pred[ctype] = max(0, pred_safe)   # valore con margine per la submission
            except Exception as e:
                print(f"Errore su {file_id} ({ctype}): {e}")
                row_pred[ctype] = 0; pure_preds[ctype] = 0

        results.append(row_pred)

    df_submission = pd.DataFrame(results).rename(columns={
        "WW": "Cycles_to_WW", "HPC": "Cycles_to_HPC_SV", "HPT": "Cycles_to_HPT_SV"
    })
    for col in ["Cycles_to_WW", "Cycles_to_HPC_SV", "Cycles_to_HPT_SV"]:
        df_submission[col] = df_submission[col].round(0).astype(int)

    df_submission = df_submission[["file", "Cycles_to_WW", "Cycles_to_HPC_SV", "Cycles_to_HPT_SV"]]
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    df_submission.to_csv(output_file, index=False)
    print(f"File salvato: {output_file}")

## 7. Caricamento dati e addestramento

Carica il training (preferendo `train_cleaned.csv`), lo armonizza, prepara i due
dataset per evento e lancia la catena Domino. Gli artefatti addestrati
(`trained_models`, `feature_sets`, `oof_predictions_dict`) vengono riutilizzati
nelle sezioni successive.

In [ ]:
print("--- CARICAMENTO TRAINING ---")
if os.path.exists(TRAIN_PRIMARY):
    df_raw = pd.read_csv(TRAIN_PRIMARY)
elif os.path.exists(TRAIN_FALLBACK):
    df_raw = pd.read_csv(TRAIN_FALLBACK)
else:
    raise FileNotFoundError(f"Nessun file di training in '{DATA_ELAB_TRAIN_DIR}'.")

df_raw = harmonize_columns(df_raw)

print("\n--- DATASET PREPARATION ---")
df_mech = prepare_mechanical_data(df_raw, is_test=False)
df_wash = prepare_wash_data(df_raw, is_test=False)

trained_models, feature_sets, oof_predictions_dict = train_domino_chain(df_mech, df_wash)

## 8. Validazione

Genera la submission sui motori di validazione (`data/val/`) applicando la catena
Domino. I punteggi OOF per evento sono stampati durante l'addestramento (Sezione 7).

In [ ]:
if os.path.isdir(VAL_DIR):
    generate_submission_file(VAL_DIR, trained_models, feature_sets, output_file=SUB_VAL_FILE)
    display(pd.read_csv(SUB_VAL_FILE).head())
else:
    print(f"[SKIP] Cartella di validazione assente: '{VAL_DIR}'")

## 9. Generazione submission finale

Genera la submission ufficiale sui motori di test/gara (`data/test/`).

In [ ]:
if os.path.isdir(TEST_DIR):
    print("--- AVVIO GENERAZIONE SUBMISSION FINALE (Test Set) ---")
    generate_submission_file(TEST_DIR, trained_models, feature_sets, output_file=SUB_FINAL_FILE)
    display(pd.read_csv(SUB_FINAL_FILE).head())
else:
    print(f"[SKIP] Cartella di test assente: '{TEST_DIR}'")

## 10. (Opzionale) Importanza delle feature del modello finale

Visualizza le feature più importanti per ciascun evento, **riutilizzando i modelli
già addestrati** nella Sezione 7 (nessun riaddestramento). Le feature fisiche/predette
(catena) sono evidenziate rispetto ai sensori grezzi.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
titles = {"WW": "Lavaggio compressore (WW)", "HPC": "Guasto compressore (HPC)", "HPT": "Guasto turbina (HPT)"}
TOPN = 12

for ax, ctype in zip(axes, ["WW", "HPC", "HPT"]):
    feats = feature_sets[ctype]
    imp = trained_models[ctype]["model"].feature_importances_
    order = np.argsort(imp)[::-1][:TOPN]
    names = [feats[i] for i in order]
    vals = [imp[i] for i in order]
    colors = ["#14459E" if (n.startswith("Phy_") or n.startswith("Pred_")) else "#9AA6BC" for n in names]
    ax.barh(range(len(names)), vals, color=colors)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=9)
    ax.invert_yaxis()
    ax.set_title(titles[ctype], fontsize=13, fontweight="bold"); ax.set_xlabel("Importanza")
    print(f"\n[{ctype}] Top 5 feature:")
    for i in order[:5]:
        print(f"   {feats[i]:35s} {imp[i]:.4f}")

fig.legend(handles=[Patch(color="#14459E", label="Feature fisiche / predette (catena)"),
                    Patch(color="#9AA6BC", label="Sensori grezzi")],
           loc="lower center", ncol=2, fontsize=11, frameon=False)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("feature_importance_modello_finale.png", dpi=200, bbox_inches="tight")
plt.show()
print("\nSalvato: feature_importance_modello_finale.png")